# SegmentEveryForam v2 fine-tuning
100 epochs from v1. The dataset is split by original image before patchifying. Select a GPU runtime and authorize your own Google Drive when prompted.

Before running, place `SEF_v2_colab_inputs.zip` in your Google Drive root. The ZIP must contain `seg_model_foram_v1_30epochs.keras` at its root and 40 matching `_image` and `_mask` files in `inputs/` (40 pairs). The v1 checkpoint is in this repository’s `models/` folder. The source micrographs and masks are supplied separately.

The run writes `SEF_v2_100epochs/best_val_loss.keras` to your Drive. That file is the lowest-validation-loss checkpoint from the 100-epoch run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, os, shutil, tempfile, zipfile
import numpy as np
from PIL import Image
from glob import glob
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import load_img

if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Select a GPU under Runtime > Change runtime type.')
keras.utils.set_random_seed(42)
root = Path('/content/drive/MyDrive')
bundle = root / 'SEF_v2_colab_inputs.zip'
run = root / 'SEF_v2_100epochs'
run.mkdir(exist_ok=True)
work = Path(tempfile.mkdtemp(prefix='sef_', dir='/content'))
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(work)
print(tf.config.list_physical_devices('GPU'))

In [ ]:
def weighted_crossentropy(y_true, y_pred, label_smoothing: float = 0.1):
    """
    Calculates the weighted cross-entropy loss between the true labels and predicted labels.

    Uses label smoothing to prevent overconfident predictions and improve
    generalization. Expects logits (pre-softmax) as y_pred.

    Parameters
    ----------
    y_true : tensor
        True labels (one-hot encoded).
    y_pred : tensor
        Predicted logits (pre-softmax).
    label_smoothing : float
        Label smoothing factor. 0.0 means no smoothing.

    Returns
    -------
    loss : tensor
        Weighted cross-entropy loss.
    """
    num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
    y_true_smooth = y_true * (1.0 - label_smoothing) + label_smoothing / num_classes
    class_weights = tf.constant(
        [[[[0.6, 1.0, 5.0]]]]
    )  # increase the weight on the grains and the grain boundaries
    unweighted_losses = tf.nn.softmax_cross_entropy_with_logits(
        labels=y_true_smooth, logits=y_pred
    )
    weights = tf.reduce_sum(class_weights * y_true, axis=-1)
    weighted_losses = weights * unweighted_losses
    loss = tf.reduce_mean(weighted_losses)
    return loss

In [ ]:
def load_and_preprocess(image_path, mask_path, augmentations=False):
    """
    Load and preprocess an image and its corresponding mask.

    Parameters
    ----------
    image_path : str
        The file path of the image.
    mask_path : str
        The file path of the mask.
    augmentations : bool, optional
        Whether to apply augmentations to the image. Defaults to False.

    Returns
    -------
    image : numpy.ndarray
        the preprocessed image.
    mask : numpy.ndarray
        the preprocessed mask.
    """
    # Load image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)

    # Load mask and one-hot encode it
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.cast(mask, tf.int32)
    mask = tf.one_hot(mask, depth=3, axis=-1)
    mask = tf.reshape(mask, (256, 256, 3))

    # Normalize images
    image = tf.cast(image, tf.float32) / 255.0

    # Apply augmentations
    if augmentations:
        # --- Geometric augmentations (applied to both image and mask) ---
        # Concatenate along channels so identical transforms apply to both
        combined = tf.concat([image, mask], axis=-1)  # shape: (256, 256, 6)

        # Random horizontal flip
        seed_hflip = tf.random.uniform(shape=[], minval=0, maxval=1)
        combined = tf.cond(
            seed_hflip < 0.5,
            lambda: tf.image.flip_left_right(combined),
            lambda: combined,
        )

        # Random vertical flip
        seed_vflip = tf.random.uniform(shape=[], minval=0, maxval=1)
        combined = tf.cond(
            seed_vflip < 0.5,
            lambda: tf.image.flip_up_down(combined),
            lambda: combined,
        )

        # Random 90-degree rotations (0, 1, 2, or 3 times)
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
        combined = tf.image.rot90(combined, k=k)

        # Random scaling/zoom (crop and resize)
        seed_scale = tf.random.uniform(shape=[], minval=0, maxval=1)
        def apply_random_scale(combined):
            scale = tf.random.uniform(shape=[], minval=0.7, maxval=0.95)
            crop_size = tf.cast(tf.round(256.0 * scale), tf.int32)
            max_offset = tf.maximum(256 - crop_size, 1)
            offset_h = tf.random.uniform(shape=[], minval=0,
                                         maxval=max_offset, dtype=tf.int32)
            offset_w = tf.random.uniform(shape=[], minval=0,
                                         maxval=max_offset, dtype=tf.int32)
            combined_cropped = tf.image.crop_to_bounding_box(
                combined, offset_h, offset_w, crop_size, crop_size
            )
            return tf.image.resize(combined_cropped, [256, 256],
                                   method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
        combined = tf.cond(
            seed_scale < 0.5, lambda: apply_random_scale(combined), lambda: combined
        )

        # Split back into image and mask
        image = combined[:, :, :3]
        mask = combined[:, :, 3:]

        # Re-discretize mask after geometric transforms (nearest-neighbor keeps it
        # clean, but this guards against any floating-point drift)
        mask = tf.round(mask)

        # --- Color augmentations (applied to image only) ---
        # Each augmentation applied independently with 50% probability

        # Random brightness
        seed_bright = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_bright < 0.5,
            lambda: tf.image.random_brightness(image, max_delta=0.2),
            lambda: image,
        )

        # Random contrast
        seed_contrast = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_contrast < 0.5,
            lambda: tf.image.random_contrast(image, lower=0.8, upper=1.2),
            lambda: image,
        )

        # Random saturation
        seed_sat = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_sat < 0.5,
            lambda: tf.image.random_saturation(image, lower=0.7, upper=1.3),
            lambda: image,
        )

        # Random hue
        seed_hue = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_hue < 0.5,
            lambda: tf.image.random_hue(image, max_delta=0.05),
            lambda: image,
        )

        # Gaussian noise
        seed_noise = tf.random.uniform(shape=[], minval=0, maxval=1)
        image = tf.cond(
            seed_noise < 0.5,
            lambda: image + tf.random.normal(tf.shape(image), mean=0.0, stddev=0.02),
            lambda: image,
        )

        # Gaussian blur
        seed_blur = tf.random.uniform(shape=[], minval=0, maxval=1)
        def apply_gaussian_blur(img):
            sigma = tf.random.uniform(shape=[], minval=0.5, maxval=1.5)
            kernel_size = 5
            ax = tf.cast(tf.range(-kernel_size // 2 + 1, kernel_size // 2 + 1),
                         tf.float32)
            xx, yy = tf.meshgrid(ax, ax)
            kernel = tf.exp(-(xx ** 2 + yy ** 2) / (2.0 * sigma ** 2))
            kernel = kernel / tf.reduce_sum(kernel)
            kernel = tf.reshape(kernel, [kernel_size, kernel_size, 1, 1])
            kernel = tf.tile(kernel, [1, 1, 3, 1])
            img = tf.expand_dims(img, axis=0)
            img = tf.nn.depthwise_conv2d(img, kernel, strides=[1, 1, 1, 1],
                                         padding='SAME')
            return tf.squeeze(img, axis=0)
        image = tf.cond(
            seed_blur < 0.3, lambda: apply_gaussian_blur(image), lambda: image
        )

        # Clip to valid range
        image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask

In [ ]:
def patchify_training_data(input_dir, patch_dir):
    """
    Extracts patches from training images and labels, and saves them to the specified directory.

    Parameters
    ----------
    input_dir : str
        The directory containing the input images and labels.
    patch_dir : str
        The directory where the patches will be saved.

    Returns
    -------
    tuple
        A tuple containing the paths to the directories where the image patches and label patches are saved.

    Notes
    -----
    - The function expects the input directory to contain files with 'image' and 'mask' in their filenames.
    - The patches are extracted with a size of 256x256 pixels and a stride of 128 pixels.
    - The function creates subdirectories 'images' and 'labels' within the specified patch directory to save the patches.
    - If the input directory does not exist or contains no matching files, the function will print a warning and return.
    """
    if not os.path.exists(input_dir):
        print(f"Warning: The directory {input_dir} does not exist.")
        return

    images = sorted(glob(input_dir + "*image*"))
    labels = sorted(glob(input_dir + "*mask*"))

    if not images or not labels:
        print(f"Warning: No files containing 'image' or 'mask' found in {input_dir}.")
        return

    # Create directories for patches
    patches_dir = os.path.join(patch_dir, "Patches")
    images_dir = os.path.join(patches_dir, "images")
    labels_dir = os.path.join(patches_dir, "labels")
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    start_no = 0
    for image in tqdm(images):
        # Load the large image
        large_image = load_img(image)
        # Convert the image to a tensor
        large_image = tf.keras.preprocessing.image.img_to_array(large_image)
        # Reshape the tensor to have a batch size of 1
        large_image = tf.reshape(large_image, [1, *large_image.shape])
        # Extract patches from the large image
        patches = tf.image.extract_patches(
            images=large_image,
            sizes=[1, 256, 256, 1],
            strides=[1, 128, 128, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        # Reshape the patches tensor to have a batch size of -1
        patches = tf.reshape(patches, [-1, 256, 256, 3])
        # Write patches to files
        for i in range(patches.shape[0]):
            im = np.asarray(patches[i, :, :, :]).astype("uint8")
            imname = os.path.join(images_dir, "im%03d.png" % (start_no + i))
            im = Image.fromarray(im.astype(np.uint8))
            im.save(imname)
        start_no = start_no + patches.shape[0]

    start_no = 0
    for image in tqdm(labels):
        # Load the large image
        large_image = load_img(image)
        # Convert the image to a tensor
        large_image = tf.keras.preprocessing.image.img_to_array(large_image)
        large_image = large_image[
            :, :, 0, np.newaxis
        ]  # only keep one layer and add a new axis
        # Reshape the tensor to have a batch size of 1
        large_image = tf.reshape(large_image, [1, *large_image.shape])
        # Extract patches from the large image
        patches = tf.image.extract_patches(
            images=large_image,
            sizes=[1, 256, 256, 1],
            strides=[1, 128, 128, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        # Reshape the patches tensor to have a batch size of -1
        patches = tf.reshape(patches, [-1, 256, 256, 1])
        # Write patches to files
        for i in range(patches.shape[0]):
            im = np.asarray(patches[i, :, :, 0]).astype("uint8")
            imname = os.path.join(labels_dir, "im%03d.png" % (start_no + i))
            im = Image.fromarray(im.astype(np.uint8))
            im.save(imname)
        start_no = start_no + patches.shape[0]
    return images_dir, labels_dir

In [ ]:
def index_files(suffix):
    result = {}
    for file in (work / 'inputs').iterdir():
        if file.stem.lower().endswith(suffix):
            key = file.stem[:-len(suffix)].lower()
            if key in result:
                raise ValueError(f'Duplicate pair key: {key}')
            result[key] = file
    return result

images, masks = index_files('_image'), index_files('_mask')
if images.keys() != masks.keys() or len(images) != 40:
    raise ValueError('Expected 40 matched original image-mask pairs.')
for key in images:
    with Image.open(images[key]) as image, Image.open(masks[key]) as mask:
        if image.size != mask.size or not np.isin(np.asarray(mask.convert('RGB'))[:,:,0], [0,1,2]).all():
            raise ValueError(f'Check mask dimensions or class labels: {key}')

train_val, test = train_test_split(sorted(images), test_size=6, random_state=42)
train, val = train_test_split(train_val, test_size=6, random_state=42)
splits = {'train': train, 'val': val, 'test': test}
manifest = 'split,image,mask\n' + ''.join(
    f'{split},{images[key].name},{masks[key].name}\n'
    for split, keys in splits.items() for key in sorted(keys)
)
manifest_path = run / 'image_split.csv'
if manifest_path.exists() and manifest_path.read_text() != manifest:
    raise ValueError('The saved split differs from these inputs.')
manifest_path.write_text(manifest)

def make_dataset(split, keys):
    source = work / split / 'originals'
    source.mkdir(parents=True, exist_ok=True)
    for i, key in enumerate(sorted(keys)):
        for label, files in [('image', images), ('mask', masks)]:
            file = files[key]
            shutil.copyfile(file, source / f'{i:04d}_{label}{file.suffix}')
    image_dir, mask_dir = map(Path, patchify_training_data(
        str(source) + os.sep, str(source.parent)))
    names = sorted(file.name for file in image_dir.glob('*.png'))
    if not names or set(names) != {file.name for file in mask_dir.glob('*.png')}:
        raise ValueError(f'Unmatched patches in {split}')
    print(f'{split}: {len(keys)} original images, {len(names)} patches')
    ds = tf.data.Dataset.from_tensor_slices(
        ([str(image_dir / n) for n in names], [str(mask_dir / n) for n in names]))
    augment = split == 'train'
    if augment:
        ds = ds.shuffle(min(1000, len(names)), seed=42)
    ds = ds.map(lambda x, y: load_and_preprocess(x, y, augment),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

datasets = {split: make_dataset(split, keys) for split, keys in splits.items()}

In [ ]:
import json

def load_checkpoint(path):
    return keras.models.load_model(path, custom_objects={
        'weighted_crossentropy': weighted_crossentropy})

def save_checkpoint(model, path):
    local = work / 'checkpoint.keras'
    model.save(local)
    pending = path.with_suffix('.pending.keras')
    shutil.copyfile(local, pending)
    os.replace(pending, path)

progress = run / 'completed.json'

class SaveProgress(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        number = epoch + 1
        loss = float(logs['val_loss'])
        slot = run / f'resume_{number % 2}.keras'
        save_checkpoint(self.model, slot)
        if number in (30, 100):
            shutil.copyfile(slot, run / f'epoch_{number:03d}.keras')
        if loss < state['best']:
            shutil.copyfile(slot, run / 'best_val_loss.keras')
            state['best'] = loss
        state['epoch'] = number
        state['logs'].append({k: float(v) for k,v in logs.items()})
        pending = run / 'completed.tmp'
        pending.write_text(json.dumps(state, indent=2))
        os.replace(pending, progress)

state = json.loads(progress.read_text()) if progress.exists() else {
    'epoch': 0, 'best': float('inf'), 'logs': []}
start = state['epoch']
source_model = (run / f'resume_{start % 2}.keras' if start else
                work / 'seg_model_foram_v1_30epochs.keras')
model = load_checkpoint(source_model)
if start < 100:
    model.fit(datasets['train'], validation_data=datasets['val'],
              initial_epoch=start, epochs=100, callbacks=[SaveProgress()])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(state['logs'], index=range(1, state['epoch'] + 1))
history[['loss', 'val_loss']].plot(xlabel='Epoch', ylabel='Weighted loss')
plt.savefig(run / 'loss.png', dpi=160, bbox_inches='tight')
history.to_csv(run / 'history.csv', index_label='epoch')
for epoch in (30, 100):
    candidate = load_checkpoint(run / f'epoch_{epoch:03d}.keras')
    print(epoch, candidate.evaluate(datasets['test'], return_dict=True))